# 📌 6️⃣ Transfer Learning

---

## 🧠 1️⃣ What is Transfer Learning?

Transfer Learning (TL) is the technique of using a **pre-trained model** (trained on a large dataset like ImageNet) for a new but related task.

---

### 🔹 Why Transfer Learning?

- Training deep CNNs from scratch needs **huge data** and **computational resources**  
- Pre-trained models have already learned useful features like **edges, shapes, textures**  
- TL allows **faster training**, **better accuracy**, and works even with **small datasets**

---

### 🔹 How It Works

**Feature Extraction:**

- Use pre-trained model as a **fixed feature extractor**  
- Freeze its layers, only train a new classifier  

**Fine-Tuning:**

- Unfreeze some top layers of pre-trained model  
- Train both classifier + top layers  
- Improves accuracy for your dataset  

---

### 🔹 Common Pre-trained Models

| Model       | Depth | Parameters | Strengths                                  |
|------------|-------|------------|--------------------------------------------|
| VGG16/19   | 16-19 | 138M       | Simple, easy to use, large memory          |
| ResNet50   | 50    | 25M        | Residual connections, very deep            |
| MobileNet  | 28    | 4M         | Lightweight, fast, mobile-friendly         |

---

## 🟢 2️⃣ Transfer Learning Example: VGG16

Task: Image classification (e.g., cats vs dogs)


In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D

# Load VGG16 base model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # Freeze base

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


### 🔹 Training

In [ ]:
history = model.fit(train_generator,
                    validation_data=validation_generator,
                    epochs=10)


✅ Notes:

- VGG16 → learns high-level features from ImageNet

- We only train top classifier first

## 🔵 3️⃣ Transfer Learning Example: ResNet50

Task: Leaf disease classification

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_generator, validation_data=validation_generator, epochs=15)

✅ Notes:

- ResNet50 → uses residual connections → prevents vanishing gradients

- Good for deep networks

## 🟣 4️⃣ Transfer Learning Example: MobileNet

Task: Mobile-friendly image classification

In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

base_model = MobileNet(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
predictions = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.fit(train_generator, validation_data=validation_generator, epochs=10)

✅ Notes:

- MobileNet → lightweight, ideal for mobile devices

- Smaller memory footprint → faster inference

## 🔹 5️⃣ Fine-Tuning Pre-trained Models

After training top layers, you can unfreeze top layers of the base model for fine-tuning:

In [ ]:
# Unfreeze last 10 layers of base model
for layer in base_model.layers[-10:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(train_generator, validation_data=validation_generator, epochs=5)

✅ Fine-tuning improves performance for your specific dataset.

## 🔹 6️⃣ Key Points for Transfer Learning

- Freeze base layers first → prevents overfitting, speeds up training  
- Use GlobalAveragePooling2D instead of Flatten → reduces parameters  
- Add Dropout → regularization  
- Fine-tune top layers only, especially for small datasets  

- VGG → simple & deep  
- ResNet → very deep, prevents gradient issues  
- MobileNet → fast & lightweight


# Combined Transfer Learning Example: Cats vs Dogs

## 🧠 1️⃣ Dataset

- Dataset: Cats vs Dogs (~25k images)

- Classes: 2 (Cat, Dog)

- Images resized to (224, 224, 3)

### 🔹 Data Generators with Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training",
)

validation_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
)

num_classes = train_generator.num_classes

### 🟢 2️⃣ VGG16 Transfer Learning

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

x = vgg_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

vgg_model = Model(inputs=vgg_base.input, outputs=predictions)
vgg_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
vgg_model.fit(train_generator, validation_data=validation_generator, epochs=10)

✅ Notes:

- Large model, many parameters → slower but strong features

- Good baseline

### 🔵 3️⃣ ResNet50 Transfer Learning

In [ ]:
from tensorflow.keras.applications import ResNet50

resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
resnet_base.trainable = False

x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

resnet_model = Model(inputs=resnet_base.input, outputs=predictions)
resnet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
resnet_model.fit(train_generator, validation_data=validation_generator, epochs=10)

✅ Notes:

- ResNet → residual connections → very deep networks

- Slightly better accuracy than VGG for complex data

### 🟣 4️⃣ MobileNet Transfer Learning

In [ ]:
from tensorflow.keras.applications import MobileNet

mobilenet_base = MobileNet(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)
mobilenet_base.trainable = False

x = mobilenet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
predictions = Dense(num_classes, activation="softmax")(x)

mobilenet_model = Model(inputs=mobilenet_base.input, outputs=predictions)
mobilenet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
mobilenet_model.fit(train_generator, validation_data=validation_generator, epochs=10)

✅ Notes:

- Lightweight → fast training & inference

- Suitable for mobile apps or edge devices

### 🔹 5️⃣ Optional: Fine-Tuning Top Layers

In [ ]:
# Example: fine-tune last 10 layers of ResNet50
for layer in resnet_base.layers[-10:]:
    layer.trainable = True

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

resnet_model.fit(train_generator, validation_data=validation_generator, epochs=5)

- Improves accuracy for dataset-specific features  
- Keep learning rate very low for fine-tuning  

---

## 🔹 6️⃣ Comparison Summary

| Model      | Parameters | Speed   | Accuracy Potential | Best Use Case                        |
|-----------|------------|--------|------------------|-------------------------------------|
| VGG16     | ~138M      | Slow   | Good             | Strong baseline, simple tasks       |
| ResNet50  | ~25M       | Medium | Very Good        | Deep networks, better for complex datasets |
| MobileNet | ~4M        | Fast   | Good             | Mobile/edge devices, fast inference |

---

## ✅ Key Takeaways

- Use **VGG** for simplicity, **ResNet** for deep and complex patterns, **MobileNet** for speed & deployment  
- **Feature extraction first, fine-tune later**  
- **Data augmentation + Dropout + BatchNorm** → prevent overfitting


## Full Project

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications import MobileNet

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

train_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training",
)

validation_generator = train_datagen.flow_from_directory(
    "data/cats_dogs",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",
)

num_classes = train_generator.num_classes


vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

x = vgg_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

vgg_model = Model(inputs=vgg_base.input, outputs=predictions)
vgg_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
vgg_model.fit(train_generator, validation_data=validation_generator, epochs=10)


resnet_base = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
resnet_base.trainable = False

x = resnet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation="softmax")(x)

resnet_model = Model(inputs=resnet_base.input, outputs=predictions)
resnet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
resnet_model.fit(train_generator, validation_data=validation_generator, epochs=10)


mobilenet_base = MobileNet(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)
mobilenet_base.trainable = False

x = mobilenet_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
predictions = Dense(num_classes, activation="softmax")(x)

mobilenet_model = Model(inputs=mobilenet_base.input, outputs=predictions)
mobilenet_model.compile(
    optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
)
mobilenet_model.fit(train_generator, validation_data=validation_generator, epochs=10)


# Example: fine-tune last 10 layers of ResNet50
for layer in resnet_base.layers[-10:]:
    layer.trainable = True

resnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

resnet_model.fit(train_generator, validation_data=validation_generator, epochs=5)